<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>
La tasa de éxito de los lanzamientos puede depender de muchos factores como la masa de la carga, el tipo de órbita, y así sucesivamente. También puede depender de la ubicación y proximidad de un sitio de lanzamiento, es decir, la posición inicial de las trayectorias de los cohetes. Encontrar un lugar óptimo para construir un sitio de lanzamiento ciertamente involucra muchos factores y con suerte podríamos descubrir algunos de ellos analizando las ubicaciones de los sitios de lanzamiento existentes.

En los laboratorios anteriores de análisis exploratorio de datos, has visualizado el conjunto de datos de lanzamientos de SpaceX usando `matplotlib` y `seaborn` y descubierto algunas correlaciones preliminares entre el sitio de lanzamiento y las tasas de éxito. En este laboratorio, realizarás análisis visuales más interactivos usando `Folium`.

## Objectives

Este laboratorio contiene las siguientes tareas:

* **TAREA 1:** Marcar todos los sitios de lanzamiento en un mapa
* **TAREA 2:** Marcar los lanzamientos exitosos/fallidos de cada sitio en el mapa
* **TAREA 3:** Calcular las distancias entre un sitio de lanzamiento y sus proximidades

Después de completar las tareas anteriores, deberías poder encontrar algunos patrones geográficos sobre los sitios de lanzamiento.

In [1]:
try:
    import pandas as pd
    import folium
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "folium"])
    import pandas as pd
    import folium

from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon


El siguiente conjunto de datos con el nombre spacex_launch_geo.csv es un conjunto de datos ampliado con la latitud y longitud añadidas para cada sitio.

In [2]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)

Now, you can take a look at what are the coordinates for each site.

In [3]:
spacex_df= spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
spacex_df.head()

,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


In [4]:
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df.head()

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Las coordenadas de arriba son solo números simples que no te pueden dar ninguna idea intuitiva sobre dónde están esos sitios de lanzamiento. Si eres muy bueno en geografía, puedes interpretar esos números directamente en tu mente. Si no, también está bien. Vamos a visualizar esas ubicaciones marcándolas en un mapa.

We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.

In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,

In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(
    location=nasa_coordinate,
    radius=1000,
    color='blue',
    fill=True,
    fill_color='blue',
    popup='NASA Johnson Space Center'
).add_child(folium.Popup('NASA Johnson Space Center'))

marker = folium.Marker(
    location=nasa_coordinate,
    icon= DivIcon(
        icon_size=(150,36),
        icon_anchor=(7,20),
        html='<div style="font-size: 12pt; color:#d35400">NASA Johnson Space Center</div>',
    )
)

site_map.add_child(circle)
site_map.add_child(marker)
display(site_map)

y deberías encontrar un pequeño círculo amarillo cerca de la ciudad de Houston y puedes acercar para ver un círculo más grande.

Now, let's add a circle for each launch site in data frame `launch_sites`

*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map

In [8]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

Ahora, puedes explorar el mapa acercando o alejando las áreas marcadas, y tratar de responder las siguientes preguntas:

* ¿Están todos los sitios de lanzamiento cerca de la línea del ecuador?
* ¿Están todos los sitios de lanzamiento muy cerca de la costa?

También intenta explicar tus hallazgos.

In [9]:
for _, launch_site in launch_sites_df.iterrows():
    coordinates = [launch_site['Lat'], launch_site['Long']]
    site_name = launch_site['Launch Site']

    circle = folium.Circle(
        location=coordinates,
        radius=1000,
        color='blue',
        fill=True,
        fill_color='blue',
        popup=site_name
    )
    circle.add_to(site_map)

    marker = folium.Marker(
        location=coordinates,
        popup=site_name,
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(7, 20),
            html=f'<div style="font-size: 12pt">{site_name}</div>'
        )
    )
    marker.add_to(site_map)

display(site_map)

## Task 2: Mark the success/failed launches for each site on the map
A continuación, intentemos mejorar el mapa añadiendo los resultados de los lanzamientos para cada sitio, y veamos qué sitios tienen altas tasas de éxito. Recuerda que el data frame spacex_df tiene registros detallados de los lanzamientos, y la columna `class` indica si este lanzamiento fue exitoso o no


In [10]:
spacex_df.head()

,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


A continuación, vamos a crear marcadores para todos los registros de lanzamientos. Si un lanzamiento fue exitoso `(class=1)`, entonces usamos un marcador verde y si un lanzamiento falló, usamos un marcador rojo `(class=0)`

Ten en cuenta que un lanzamiento solo ocurre en uno de los cuatro sitios de lanzamiento, lo que significa que muchos registros de lanzamientos tendrán las mismas coordenadas exactas. Los grupos de marcadores pueden ser una buena manera de simplificar un mapa que contiene muchos marcadores con las mismas coordenadas.

In [11]:
marker_cluster = MarkerCluster()
# todo : Create a new column in spacex_df dataframe called marker_color to store the marker colors based on the class value
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')
# For each launch result in spacex_df data frame, add a folium.Marker to marker_cluster

site_map.add_child(marker_cluster)
for idx, row in spacex_df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=row['Launch Site'],
        icon=folium.Icon(color=row['marker_color'])
    ).add_to(marker_cluster)

site_map

A partir de los marcadores con colores en los grupos de marcadores, deberías poder identificar fácilmente qué sitios de lanzamiento tienen tasas de éxito relativamente altas.

# TASK 3: Calculate the distances between a launch site to its proximities

Primero agreguemos una `MousePosition` en el mapa para obtener las coordenadas cuando pases el ratón sobre un punto en el mapa. De esta manera, mientras exploras el mapa, puedes encontrar fácilmente las coordenadas de cualquier punto de interés (como una vía férrea).

In [12]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = folium.plugins.MousePosition(
    position='topright',
    separator=' | ',
    empty_string='NaN',
    lng_first=True,
    num_digits=5,
    prefix='Coordinates:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)
site_map

Ahora haz zoom en un sitio de lanzamiento y explora su proximidad para ver si puedes encontrar fácilmente algún ferrocarril, carretera, costa, etc. Mueve tu ratón a estos puntos y anota sus coordenadas (mostradas en la parte superior izquierda) para calcular la distancia al sitio de lanzamiento.

In [6]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Marca un punto en la costa más cercana usando MousePosition y calcula la distancia entre el punto de la costa y el sitio de lanzamiento.

In [ ]:
launch_site = launch_sites_df.iloc[0]
lat1 = launch_site['Lat']
lon1 = launch_site['Long']

# Choose a nearby point such as a city, railway, highway, coast, etc.
# Use MousePosition to get the real coordinates you want.
closest_point = [28.4865, -80.5653]  # example point near the launch site
lat2, lon2 = closest_point

distance_coordinates = calculate_distance(lat1, lon1, lat2, lon2)
print(f"Distance to selected point: {distance_coordinates:.2f} km")

# Create a marker at the selected nearby point and show the distance
folium.Marker(
    location=closest_point,
    icon=folium.DivIcon(
        html=f'<div style="font-size: 12pt; color: black;">{distance_coordinates:.2f} km</div>'
    )
).add_to(site_map)

# Draw a line between the launch site and the selected point
folium.PolyLine(
    locations=[[lat1, lon1], closest_point],
    color='blue',
    weight=2.5,
    opacity=1
).add_to(site_map)

site_map

8.513370461181905


In [9]:
# Build the final map from all launch data in spacex_df
map_center = [spacex_df['Lat'].mean(), spacex_df['Long'].mean()]
site_map = folium.Map(location=map_center, zoom_start=4)

# Add every launch site from the database
for _, launch_site in launch_sites_df.iterrows():
    launch_coordinates = [launch_site['Lat'], launch_site['Long']]
    site_name = launch_site['Launch Site']

    folium.Circle(
        location=launch_coordinates,
        radius=1000,
        color='blue',
        fill=True,
        fill_color='blue',
        popup=site_name
    ).add_to(site_map)

    folium.Marker(
        location=launch_coordinates,
        popup=site_name,
        icon=DivIcon(
            icon_size=(150, 36),
            icon_anchor=(7, 20),
            html=f'<div style="font-size: 12pt">{site_name}</div>'
        )
    ).add_to(site_map)

# Add every launch result from the database
marker_cluster = MarkerCluster().add_to(site_map)
for _, launch in spacex_df.iterrows():
    marker_color = 'green' if launch['class'] == 1 else 'red'
    folium.Marker(
        location=[launch['Lat'], launch['Long']],
        popup=f"{launch['Launch Site']} - {'Successful' if launch['class'] == 1 else 'Failed'}",
        icon=folium.Icon(color=marker_color)
    ).add_to(marker_cluster)

# Reference cities for the regions represented in the launch-site database
closest_cities = {
    'CCAFS LC-40': ('Cape Canaveral', [28.3922, -80.6077]),
    'CCAFS SLC-40': ('Cape Canaveral', [28.3922, -80.6077]),
    'KSC LC-39A': ('Titusville', [28.6122, -80.8076]),
    'VAFB SLC-4E': ('Lompoc', [34.6391, -120.4579])
}

for _, launch_site in launch_sites_df.iterrows():
    site_name = launch_site['Launch Site']
    if site_name not in closest_cities:
        continue

    city_name, city_coordinates = closest_cities[site_name]
    launch_coordinates = [launch_site['Lat'], launch_site['Long']]
    distance_to_city = calculate_distance(
        launch_coordinates[0], launch_coordinates[1],
        city_coordinates[0], city_coordinates[1]
    )

    folium.Marker(
        location=city_coordinates,
        popup=(
            f'<b>{city_name}</b><br>'
            f'Distance from {site_name}: {distance_to_city:.2f} km'
        ),
        tooltip=f'{city_name}: {distance_to_city:.2f} km',
        icon=folium.Icon(color='green', icon='info-sign')
    ).add_to(site_map)

    folium.PolyLine(
        locations=[launch_coordinates, city_coordinates],
        color='darkgreen',
        weight=3,
        opacity=0.8,
        tooltip=f'{distance_to_city:.2f} km to {city_name}'
    ).add_to(site_map)

# Keep the coordinate inspector available for exploring nearby features
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
site_map.add_child(MousePosition(
    position='topright',
    separator=' | ',
    empty_string='NaN',
    lng_first=True,
    num_digits=5,
    prefix='Coordinates:',
    lat_formatter=formatter,
    lng_formatter=formatter
))

site_map

## Conclusiones

### ¿Están los sitios de lanzamiento cerca de las vías del tren?

El mapa no incluye una capa de vías ferroviarias ni la base `spacex_df` contiene sus coordenadas, por lo que no se puede calcular una distancia exacta con estos datos. La herramienta `MousePosition` permite localizar una vía en el mapa y añadir sus coordenadas para repetir el cálculo con `calculate_distance`.

### ¿Están los sitios de lanzamiento cerca de las autopistas?

La base tampoco contiene las geometrías de las autopistas. Visualmente, los sitios de lanzamiento están conectados con la red viaria regional, pero esta observación no equivale a una distancia calculada. Para responderla numéricamente habría que seleccionar una autopista cercana con `MousePosition` o cargar una capa geográfica de carreteras.

### ¿Están los sitios de lanzamiento cerca de la costa?

Sí. Los cuatro sitios aparecen junto a la costa: los sitios de Florida están en la zona costera de Cape Canaveral y el sitio VAFB SLC-4E está en la costa de California. Esta ubicación es coherente con los lanzamientos espaciales, porque permite orientar las trayectorias sobre el océano y reduce los riesgos para las zonas pobladas.

### ¿Mantienen los sitios de lanzamiento cierta distancia de las ciudades?

Sí. Según las distancias calculadas en el mapa, los sitios están separados de las ciudades de referencia por aproximadamente **14-19 km**:

- `CCAFS LC-40` → Cape Canaveral: **19.15 km**
- `CCAFS SLC-40` → Cape Canaveral: **19.26 km**
- `KSC LC-39A` → Titusville: **16.28 km**
- `VAFB SLC-4E` → Lompoc: **14.01 km**

Por tanto, los sitios no están en el centro de las ciudades, pero tampoco se encuentran completamente aislados. Mantienen una separación moderada y siguen teniendo acceso a infraestructuras cercanas.